In [37]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn import svm
from sklearn.metrics import f1_score
import numpy as np
import neat
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from scipy.special import softmax
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
import unicodedata
import re
from sklearn.decomposition import TruncatedSVD


In [3]:
_STOPWORDS = stopwords.words("spanish")  # agregar más palabras a esta lista si es necesario
PUNCTUACTION = ";:,.\\-\"'/"
SYMBOLS = "()[]¿?¡!{}~<>|@#"
NUMBERS= "0123456789"
SKIP_SYMBOLS = set(PUNCTUACTION + SYMBOLS)
SKIP_SYMBOLS_AND_SPACES = set(PUNCTUACTION + SYMBOLS + '\t\n\r ')

def normaliza_texto(input_str,
                    punct=False,
                    accents=False,
                    num=False,
                    max_dup=2):
    """
        punct=False (elimina la puntuación, True deja intacta la puntuación)
        accents=False (elimina los acentos, True deja intactos los acentos)
        num= False (elimina los números, True deja intactos los acentos)
        max_dup=2 (número máximo de símbolos duplicados de forma consecutiva, rrrrr => rr)
    """
    
    nfkd_f = unicodedata.normalize('NFKD', input_str)
    n_str = []
    c_prev = ''
    cc_prev = 0
    for c in nfkd_f:
        if not num:
            if c in NUMBERS:
                continue
        if not punct:
            if c in SKIP_SYMBOLS:
                continue
        if not accents and unicodedata.combining(c):
            continue
        if c_prev == c:
            cc_prev += 1
            if cc_prev >= max_dup:
                continue
        else:
            cc_prev = 0
        n_str.append(c)
        c_prev = c
    texto = unicodedata.normalize('NFKD', "".join(n_str))
    texto = re.sub(r'(\s)+', r' ', texto.strip(), flags=re.IGNORECASE)
    return texto

def mi_preprocesamiento(texto):
    #convierte a minúsculas el texto antes de normalizar
    # print("antes: ", texto)
    texto = normaliza_texto(texto.lower())
    # print("después:",texto)
    return texto

def mi_tokenizador(texto):
    # Elimina stopwords: palabras que no se consideran de contenido y que no agregan valor semántico al texto
    #print("antes: ", texto)
    texto = [t for t in texto.split() if t not in _STOPWORDS]
    #print("después:",texto)
    return texto

In [38]:
dataset_train = pd.read_json("./data/dataset_polaridad_es_train_embeddings.json", lines=True)
dataset_test = pd.read_json("./data/dataset_polaridad_es_test_embeddings.json", lines=True)

In [39]:

X_train = dataset_train["text"].to_numpy()
Y_train = dataset_train["klass"].to_numpy()
X_test = dataset_test["text"].to_numpy()
Y_test = dataset_test["klass"].to_numpy()

In [ ]:

le = LabelEncoder()

Y_encoded= le.fit_transform(Y_train)
Y_test_encoded = le.transform(Y_test)

vec_tfidf = CountVectorizer(analyzer="word", preprocessor=mi_preprocesamiento, tokenizer=mi_tokenizador,  ngram_range=(1,1), max_features=3000, min_df=2, max_df=0.9)
X_tfidf = vec_tfidf.fit_transform(X_train)
X_test_tfidf = vec_tfidf.transform(X_test)

d:\Apps\envs\RNA\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [41]:
svd = TruncatedSVD(
    n_components=150,
    random_state=42
)

X_reduced = svd.fit_transform(X_tfidf)
X_test_reduced = svd.transform(X_test_tfidf)

In [42]:
X_train, X_val, Y_train_encoded, Y_val_encoded = train_test_split(
    X_reduced,
    Y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=Y_encoded
)


In [43]:
def eval_genomes(genomes, config):

    BATCH_SIZE = 512

    indices = np.random.choice(len(X_train), BATCH_SIZE, replace=False)

    X_batch = X_train[indices]
    Y_batch = Y_train_encoded[indices]

    for genome_id, genome in genomes:

        net = neat.nn.FeedForwardNetwork.create(genome, config)

        predictions = []

        for xi in X_batch:
            output = net.activate(xi)
            pred = np.argmax(output)
            predictions.append(pred)

        fitness = f1_score(
            Y_batch,
            predictions,
            average="macro"
        )
        complexity_penalty = len(genome.connections) * 0.0005

        genome.fitness = fitness - complexity_penalty

In [44]:
def run_neat(config_path):
    # Cargar configuración
    config = neat.Config(neat.DefaultGenome, neat.DefaultReproduction,
                         neat.DefaultSpeciesSet, neat.DefaultStagnation,
                         config_path)

    # Crear la población
    p = neat.Population(config)

    # Agregar reporteros para ver el progreso en consola
    p.add_reporter(neat.StdOutReporter(True))
    stats = neat.StatisticsReporter()
    p.add_reporter(stats)

    # Ejecutar por N generaciones
    winner = p.run(eval_genomes, 50) 
    
    return winner, config


In [ ]:

# Ejecución
winner_genome, config_used = run_neat('config-feedforward.cfg')


 ****** Running generation 0 ****** 

Population's average fitness: 0.16622 stdev: 0.06248
Best fitness: 0.34943 - size: (3, 90) - species 11 - id 508
Average adjusted fitness: 0.095
Mean genetic distance 2.469, standard deviation 0.337
Population of 550 members in 91 species (after reproduction):
   ID   age  size   fitness   adj fit  stag
  ====  ===  ====  =========  =======  ====
     1    0     1      0.241    0.134     0
     2    0    14      0.235    0.111     0
     3    0    11      0.221    0.083     0
     4    0    12      0.260    0.130     0
     5    0     3      0.154    0.014     0
     6    0    12      0.300    0.129     0
     7    0    16      0.301    0.078     0
     8    0     3      0.244    0.068     0
     9    0     8      0.192    0.080     0
    10    0     6      0.195    0.028     0
    11    0    30      0.349    0.127     0
    12    0    11      0.217    0.079     0
    13    0     4      0.198    0.067     0
    14    0     7      0.294    0.147   

In [35]:
# Reconstruir la mejor red
winner_net = neat.nn.FeedForwardNetwork.create(winner_genome, config_used)

# Predecir en el conjunto de TEST
test_predictions = []

for xi in X_test_reduced:

    output = winner_net.activate(xi)

    probs = softmax(output)

    pred = np.argmax(probs)

    test_predictions.append(pred)

# Calcular métrica final para el reporte
final_score = f1_score(Y_test_encoded, test_predictions, average="macro")
print(f"F1-Score de la mejor red NEAT en Test: {final_score}")

F1-Score de la mejor red NEAT en Test: 0.4038275878683349
